# 04 · Stage analysis of the parameter search

Figures for one stage (`STAGE`); computation is in `src/viz.py` and `src/sweep_inspect.py`.

- All metrics use the MSB validation records excluding U2OS: 15 records (one per cell line), 24 windows each, 360 windows.
- The figures are descriptive. Configurations are selected only by the pre-specified ranking in `configs/sweep.yml`.
- Section 5 loads checkpoints and needs TensorFlow; per-window results are cached in `per_window.csv`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists() and (p / "configs").exists())
sys.path.insert(0, str(ROOT))
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")   # analysis on CPU: never takes a GPU from running training
from src import sweep as sw, viz

pd.set_option("display.width", 200, "display.max_columns", 40)
%matplotlib inline
STAGE = "s1"
df = sw.collect(STAGE)
ranked = sw.rank(df)
cols = ["config_name", "rank_sum", "recon_r2", "control_pass", "pass_amp", "pass_ctrl", "pass_shape",
        "amp_ratio", "shape_keep", "best_val_OrigRecLoss", "epochs_run"]
ranked[[c for c in cols if c in ranked]].round(4)

## 1. Learning curves
Colour = cut-off range, line style = data mix, width = latent size; running minimum of the validation loss.

In [ ]:
viz.curves_by_factor(STAGE, "val_OrigRecLoss");
viz.curves_by_factor(STAGE, "val_FeatRecLoss");

## 2. Factor effects
Bars = mean per level, points = individual runs.

In [ ]:
viz.factor_effects(df, name=STAGE + "_factor_effects");

## 3. Reconstruction vs θ control

In [ ]:
viz.tradeoff(df, STAGE);

## 4. θ-control pass rates by condition
Amplitude ratio ≥ 1.5, monotonic in θ_L, shape preserved.

In [ ]:
viz.pass_breakdown(df, STAGE);

## 5. Per cell line (window-level recomputation)

In [ ]:
from src import sweep_inspect as si
pw = si.all_per_window(STAGE)
order = ranked["config_name"].tolist()
viz.cell_line_heatmap(pw, "control_pass", STAGE, order=order);
viz.cell_line_heatmap(pw, "recon_r2", STAGE, order=order);

θ control as a function of input amplitude (window SD tertiles).

In [ ]:
pw["input_sd_tertile"] = pd.qcut(pw["input_sd"], 3, labels=["low", "mid", "high"])
pw.pivot_table(index="config_name", columns="input_sd_tertile", values="control_pass", aggfunc="mean", observed=True).reindex(order).dropna(how="all").round(2)

## 6. Generation examples
Grey = input, dashed black = reconstruction, colour = generation with θ_L from 0.05 to 0.95 (purple → yellow) at fixed z. The same windows are shown for three configurations.

In [ ]:
ex = si.examples(STAGE, ["s1_MULTI_FC12_Z10_s0", "s1_POP_FC8_Z30_s0", "s1_POP_FC12_Z30_s0"], n_windows=3)
viz.plot_examples(ex, STAGE);

## Notes

- Validation has one record per cell line, so per-cell-line values are single-record results.
- Single seed per configuration in `s1`; factor effects are indicative only.